# Exploratory Data Analysis — REES46 Heterogeneous Graph Dataset

**Data source:** `data/processed/temporal/`

**Sections:**
1. Graph Basic Statistics
2. Interaction Distribution (Class Imbalance)
3. Node Degree Distribution (Power-law Property)
4. User Purchase Sequence Length (Temporal Split)
5. Temporal Distribution of Interactions
6. Summary Table for Paper


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

# Cau hinh Matplotlib theo chuan bao cao hoc thuat
plt.rcParams.update({
    'font.family'       : 'serif',
    'font.serif'        : ['Times New Roman', 'DejaVu Serif', 'Palatino', 'serif'],
    'font.size'         : 11,
    'axes.titlesize'    : 12,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'xtick.labelsize'   : 10,
    'ytick.labelsize'   : 10,
    'legend.fontsize'   : 10,
    'figure.dpi'        : 150,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'axes.linewidth'    : 0.8,
    'lines.linewidth'   : 1.4,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

# Dinh nghia cac duong dan du lieu (global temporal split pipeline)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED    = os.path.join(PROJECT_ROOT, 'data', 'processed', 'temporal')
EDGE_DIR     = PROCESSED                                        # train edge .npy files
NODE_MAP_DIR = os.path.join(PROCESSED, 'node_mappings')        # mappings + relation parquets
RAW_CSV_DIR  = os.path.join(PROJECT_ROOT, 'data', 'raw')       # raw CSV for temporal plot
FIG_DIR      = os.path.join(PROJECT_ROOT, 'notebooks', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Processed dir: {PROCESSED}')
print(f'Figures dir  : {FIG_DIR}')


## 1. Graph Basic Statistics

Thong ke so luong nut va canh cua Heterogeneous Graph.
Do thua cua do thi hanh vi (Behavioral Graph Sparsity) duoc dinh nghia la:

$$\text{Sparsity}_{\tau} = 1 - \frac{|\mathcal{E}_{\tau}|}{|\mathcal{U}| \times |\mathcal{I}|}$$

trong do $\tau \in \{\text{view, cart, purchase}\}$ va $|\mathcal{E}_{\tau}|$ la tong so luong tuong tac thuoc loai $\tau$.

In [ ]:
# --- Node counts ---
with open(os.path.join(PROCESSED, 'node_counts.json'), 'r') as f:
    node_counts = json.load(f)

N_U = node_counts['user']
N_P = node_counts['product']
N_C = node_counts['category']
N_B = node_counts['brand']
N_total = N_U + N_P + N_C + N_B

# --- Train edge counts (loaded mmap for speed) ---
E_view_train     = len(np.load(os.path.join(EDGE_DIR, 'view_train_src.npy'),     mmap_mode='r'))
E_cart_train     = len(np.load(os.path.join(EDGE_DIR, 'cart_train_src.npy'),     mmap_mode='r'))
E_purchase_train = len(np.load(os.path.join(EDGE_DIR, 'purchase_train_src.npy'), mmap_mode='r'))

# --- Temporal-split val/test purchase pairs ---
val_user_idx  = np.load(os.path.join(PROCESSED, 'val_user_idx.npy'))
val_prod_idx  = np.load(os.path.join(PROCESSED, 'val_product_idx.npy'))
test_user_idx = np.load(os.path.join(PROCESSED, 'test_user_idx.npy'))
test_prod_idx = np.load(os.path.join(PROCESSED, 'test_product_idx.npy'))
n_val  = len(val_user_idx)
n_test = len(test_user_idx)

# Total behavioral edges (train + val/test held-out purchases)
E_view     = E_view_train
E_cart     = E_cart_train
E_purchase = E_purchase_train + n_val + n_test

# --- Relation edges from node_mappings ---
pc_df = pd.read_parquet(os.path.join(NODE_MAP_DIR, 'product_category.parquet'))
pb_df = pd.read_parquet(os.path.join(NODE_MAP_DIR, 'product_brand.parquet'))
E_belongsTo  = len(pc_df)
E_producedBy = len(pb_df)

E_behavior_total = E_view + E_cart + E_purchase

# --- Sparsity ---
max_pairs         = N_U * N_P
sparsity_view     = 1.0 - E_view     / max_pairs
sparsity_cart     = 1.0 - E_cart     / max_pairs
sparsity_purchase = 1.0 - E_purchase / max_pairs

# --- Temporal split summary ---
eval_users_val  = len(np.unique(val_user_idx))
eval_users_test = len(np.unique(test_user_idx))

print('Node Statistics')
print(f'  User     : {N_U:,}')
print(f'  Product  : {N_P:,}')
print(f'  Category : {N_C:,}')
print(f'  Brand    : {N_B:,}')
print(f'  Total    : {N_total:,}')
print()
print('Edge Statistics (all splits combined)')
print(f'  view       : {E_view:,}  (sparsity = {sparsity_view:.6f})')
print(f'  cart       : {E_cart:,}  (sparsity = {sparsity_cart:.6f})')
print(f'  purchase   : {E_purchase:,}  (sparsity = {sparsity_purchase:.6f})')
print(f'  belongsTo  : {E_belongsTo:,}  (Product -> Category)')
print(f'  producedBy : {E_producedBy:,}  (Product -> Brand)')
print(f'  Behavior total: {E_behavior_total:,}')
print()
print('Temporal Split')
print(f'  Val  pairs: {n_val:,}  (unique users: {eval_users_val:,})')
print(f'  Test pairs: {n_test:,}  (unique users: {eval_users_test:,})')


**Nhận xét:**

- Với chỉ số Sparsity cực cao ($> 0.999$), tập dữ liệu đối mặt với thách thức nghiêm trọng về sự thiếu hụt tín hiệu tích cực (Positive signal). Việc không có tương tác không đồng nghĩa với việc người dùng không thích sản phẩm. Do đó, nhóm áp dụng Popularity-biased Negative Sampling nhằm định hình không gian biểu diễn (Latent space). Việc chọn các mẫu 'đối nghịch' dựa trên độ phổ biến giúp thiết lập ranh giới quyết định chặt chẽ hơn, buộc mô hình phải học các đặc trưng tinh vi thay vì chỉ dựa vào xu hướng đám đông

- Số lượng Category (14) rất ít so với Product (311k) cho thấy Category sẽ giúp kết nối các sản phẩm rời rạc. Việc đưa belongsTo và producedBy vào đồ thị giúp model học được tính tương đồng về đặc trưng, hữu ích cho vấn đề cold start

- Tỷ lệ View : Cart : Purchase tương đương khoảng $28 : 2 : 1$. Nếu chỉ training trên purchase, model sẽ không đủ dữ liệu. Các hành vi view và cart đóng vai trò là tín hiệu giám sát yếu (Weak supervision signals) để hỗ trợ dự đoán hành vi mua hàng chính xác hơn.

## 2. Interaction Distribution (Class Imbalance)

Phân tích tỷ lệ phân bố giữa ba loại hành vi: **view**, **cart**, **purchase**.

In [ ]:
behavior_counts = {
    'View': E_view,
    'Cart': E_cart,
    'Purchase': E_purchase,
}
total_beh = sum(behavior_counts.values())
behavior_pct = {k: v / total_beh * 100 for k, v in behavior_counts.items()}

for beh, cnt in behavior_counts.items():
    print(f'{beh}: {cnt} ({behavior_pct[beh]:.2f}%)')
print(f'{"Total"}: {total_beh}  (100.00%)')

In [ ]:
labels = list(behavior_counts.keys())
counts = [behavior_counts[k] for k in labels]
pcts = [behavior_pct[k] for k in labels]

colors = ['#2c6fad', '#5ba4cf', '#a8d5f5']
hatches = ['', '//', '..']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

bars = ax1.bar(labels, counts, color=colors, edgecolor='#1a1a1a',
               linewidth=0.8, zorder=3)
ax1.set_yscale('log')
ax1.set_ylabel('Number of Interactions (log scale)')
ax1.set_title('Behavior Interaction Count (Log Scale)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x:,.0f}'
))

for bar, cnt, pct in zip(bars, counts, pcts):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        cnt * 1.3,
        f'{cnt/1e6:.1f}M\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=9
    )

ax1.set_ylim(top=max(counts) * 15)
ax1.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4, zorder=0)
sns.despine(ax=ax1)

wedge_props = dict(linewidth=0.8, edgecolor='white')
wedges, texts, autotexts = ax2.pie(
    counts,
    labels=None,
    autopct=lambda p: f'{p:.1f}%',
    startangle=90,
    colors=colors,
    wedgeprops=wedge_props,
    pctdistance=0.75,
    explode=[0, 0.04, 0.08],
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')

ax2.legend(
    wedges,
    [f'{lbl}  ({cnt/1e6:.1f}M)' for lbl, cnt in zip(labels, counts)],
    loc='lower center',
    bbox_to_anchor=(0.5, -0.12),
    ncol=1,
    frameon=False,
    fontsize=10,
)
ax2.set_title('Proportion of Behavior Types')

plt.suptitle(
    'REES46 Dataset: Interaction Distribution across Behavior Types',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_interaction_distribution.pdf'))
plt.savefig(os.path.join(FIG_DIR, 'fig_interaction_distribution.png'))
plt.show()

**Nhận xét:**

- Quan sát biểu đồ, ta thấy hành vi View chiếm tới 89.9% tổng dữ liệu, trong khi Purchase chỉ chiếm một phần rất nhỏ 3.2% nên nếu sử dụng Loss BPR thông thường cho tất cả các loại cạnh thì tín hiệu gradient từ hành vi view sẽ lấn át hành vi purchase khi đó khó dự đoán được hành vi mua của người dùng.

- Bằng cách gán trọng số lớn hơn cho các hành vi hiếm (Purchase) và giảm trọng số cho các hành vi phổ biến (View), MultiTaskBPRLoss đảm bảo rằng mỗi tương tác mua hàng có đóng góp đáng kể vào việc cập nhật trọng số mô hình

- Nhóm thiết lập trọng số cho từng tác vụ (task weight) theo công thức: $w_b \propto 1/\sqrt{N_b}$ để giúp trọng số không bị đẩy lên quá cao đối với những hành vi cực hiếm, từ đó giữ cho quá trình huấn luyện ổn định hơn.

## 3. Node Degree Distribution (Power-law Property)

Phan phoi bac (degree distribution) cua User va Product tren do thi hanh vi.
Do thi TMDT thuong tuan theo dinh luat luy thua (power-law),
bieu hien qua duong thang tren truc log-log:

$$P(k) \propto k^{-\alpha}$$

In [ ]:
all_user_src = []
all_prod_dst = []

for beh in ['view', 'cart', 'purchase']:
    src = np.load(os.path.join(EDGE_DIR, f'{beh}_train_src.npy'), mmap_mode='r')
    dst = np.load(os.path.join(EDGE_DIR, f'{beh}_train_dst.npy'), mmap_mode='r')
    all_user_src.append(src)
    all_prod_dst.append(dst)
    print(f'{beh}_train: {len(src):,} canh')

# Gop cac mang lai, dem bac cua moi nut
user_src_all = np.concatenate(all_user_src)
prod_dst_all = np.concatenate(all_prod_dst)

# bincount: mang dem so lan xuat hien cua moi node id
user_degree = np.bincount(user_src_all, minlength=N_U)
prod_degree = np.bincount(prod_dst_all, minlength=N_P)

# Chi giu cac nut co bac > 0
user_deg_nonzero = user_degree[user_degree > 0]
prod_deg_nonzero = prod_degree[prod_degree > 0]

print(f'User degree (train): active users={len(user_deg_nonzero):,}')
print(f'mean={user_deg_nonzero.mean():.1f}, median={np.median(user_deg_nonzero):.1f}, '
      f'max={user_deg_nonzero.max():,}, min={user_deg_nonzero.min()}')

print(f'Product degree (train): active products={len(prod_deg_nonzero):,}')
print(f'mean={prod_deg_nonzero.mean():.1f}, median={np.median(prod_deg_nonzero):.1f}, '
      f'max={prod_deg_nonzero.max():,}, min={prod_deg_nonzero.min()}')


In [ ]:
# Ve bieu do phan phoi bac tren truc log-log
# Dung Complementary CDF (CCDF): P(Degree >= k) de duong bien dep hon tren log-log

def compute_ccdf(degrees: np.ndarray):
    # Sap xep cac gia tri bac duy nhat
    sorted_deg = np.sort(np.unique(degrees))
    n = len(degrees)
    # P(X >= k) = so nut co bac >= k / tong so nut
    ccdf = np.array([(degrees >= k).sum() / n for k in sorted_deg])
    return sorted_deg, ccdf

# Tinh CCDF cho User va Product
# Su dung subsample de tang toc (neu tap qua lon)
rng = np.random.default_rng(seed=42)

MAX_POINTS = 300  

def sample_for_plot(degrees, max_pts=MAX_POINTS):
    uniq = np.unique(degrees)
    if len(uniq) <= max_pts:
        return compute_ccdf(degrees)
    # Lay mau log-uniform cac gia tri bac
    log_idx = np.unique(np.round(
        np.logspace(0, np.log10(len(uniq) - 1), max_pts)
    ).astype(int))
    log_idx = np.clip(log_idx, 0, len(uniq) - 1)
    sampled_deg = uniq[log_idx]
    n = len(degrees)
    ccdf = np.array([(degrees >= k).sum() / n for k in sampled_deg])
    return sampled_deg, ccdf

u_deg_x, u_ccdf = sample_for_plot(user_deg_nonzero)
p_deg_x, p_ccdf = sample_for_plot(prod_deg_nonzero)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# --- User degree distribution ---
ax1.loglog(u_deg_x, u_ccdf, marker='.', markersize=3,
           linestyle='-', linewidth=1.0, color='#2c6fad', label='User (view+cart+purchase)')
ax1.set_xlabel('Degree $k$')
ax1.set_ylabel('$P(\\mathrm{Degree} \\geq k)$')
ax1.set_title('User Degree Distribution (CCDF)')
ax1.legend(frameon=False)
ax1.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.4)
sns.despine(ax=ax1)
# Them chu thich phan Long-tail
ax1.annotate(
    'Long-tail region',
    xy=(u_deg_x[-20], u_ccdf[-20]),
    xytext=(u_deg_x[-20] * 0.15, u_ccdf[-20] * 8),
    arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
    fontsize=9, color='black'
)

# --- Product degree distribution ---
ax2.loglog(p_deg_x, p_ccdf, marker='.', markersize=3,
           linestyle='-', linewidth=1.0, color='#c0392b', label='Product (view+cart+purchase)')
ax2.set_xlabel('Degree $k$')
ax2.set_ylabel('$P(\\mathrm{Degree} \\geq k)$')
ax2.set_title('Product Degree Distribution (CCDF)')
ax2.legend(frameon=False)
ax2.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.4)
sns.despine(ax=ax2)

plt.suptitle(
    'Degree Distribution on REES46 Behavioral Graph (Train Split, Log-Log Scale)',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_degree_distribution.pdf'))
plt.savefig(os.path.join(FIG_DIR, 'fig_degree_distribution.png'))
plt.show()

**Nhận xét**:
- Biểu đồ Log-Log CCDF xác nhận cấu trúc Power-law của dữ liệu, nơi 10% sản phẩm chiếm 81% tương tác. Đa số User và Product nằm ở phía bên trái biểu đồ (bậc thấp) -> Vấn đề long tail
- Trong khi đó, 50% người dùng (Median) chỉ có dưới 36 tương tác trên toàn bộ các hành vi. Sự chênh lệch khổng lồ giữa nhóm 'Head' và vùng 'Long-tail' này là minh chứng thực nghiệm cho việc phải áp dụng Popularity-biased Sampling và Multi-task Learning nhằm đảm bảo mô hình không bị thiên kiến và có thể hoạt động tốt trên cả những người dùng có ít dữ liệu.

## 4. User Purchase Sequence Length (LOO Split Justification)

Phan tich so luong giao dich **purchase** cua moi nguoi dung tren toan bo tap du lieu.
Ket qua nay giai thich tai sao **Protocol A (Leave-One-Out)** la phu hop:

- Nhung nguoi dung chi co **1 lan mua** khong the tham gia tap danh gia (thieu tap val).
- Phan bo cho thay da so nguoi dung co rat it luot mua (Long-tail), dieu nay doi hoi mo hinh
  phai khai thac tot cac hanh vi phu tro (view, cart) de hieu nguoi dung.

In [ ]:
# Train purchase src
pu_train_src = np.load(os.path.join(EDGE_DIR, 'purchase_train_src.npy'), mmap_mode='r')
print(f'purchase_train : {len(pu_train_src):,} canh')
print(f'purchase_val   : {n_val:,} pairs (temporal split)')
print(f'purchase_test  : {n_test:,} pairs (temporal split)')

# Gop tat ca purchase per user (train src + val/test user idx)
all_purchase_src = np.concatenate([pu_train_src, val_user_idx, test_user_idx])
user_purchase_deg = np.bincount(all_purchase_src.astype(np.int64), minlength=N_U)
del pu_train_src, all_purchase_src

pu_nonzero = user_purchase_deg[user_purchase_deg > 0]

n_single_purchase = (user_purchase_deg == 1).sum()
n_two_plus        = (user_purchase_deg >= 2).sum()

print(f'\nUsers co >= 1 purchase : {len(pu_nonzero):,}')
print(f'Users co == 1 purchase : {n_single_purchase:,}  '
      f'({n_single_purchase/len(pu_nonzero)*100:.1f}%)')
print(f'Users co >= 2 purchase : {n_two_plus:,}  '
      f'({n_two_plus/len(pu_nonzero)*100:.1f}%)')
print(f'Median purchases/user  : {np.median(pu_nonzero):.0f}')
print(f'Mean   purchases/user  : {pu_nonzero.mean():.2f}')
print(f'Max    purchases/user  : {pu_nonzero.max():,}')


In [ ]:
MAX_DISPLAY = 30
pu_clipped = np.clip(pu_nonzero, 1, MAX_DISPLAY)
counts_arr = np.bincount(pu_clipped, minlength=MAX_DISPLAY + 1)[1:]  # bo index 0
x_vals = np.arange(1, MAX_DISPLAY + 1)

fig, ax = plt.subplots(figsize=(9, 4))

# Mau: phan biet ro nhom 1 purchase (bi loai khoi LOO) va nhom >= 2
bar_colors = ['#c0392b'] + ['#2c6fad'] * (MAX_DISPLAY - 1)
ax.bar(x_vals, counts_arr, color=bar_colors, edgecolor='white', linewidth=0.4)

# Duong ke doc phan cach nhom chi co 1 purchase
ax.axvline(x=1.5, color='black', linestyle='--', linewidth=1.0)
ax.text(1.55, counts_arr.max() * 0.92,
        'Excluded from\nLOO evaluation\n(only 1 purchase)',
        fontsize=8.5, color='#c0392b', va='top')

ax.set_xlabel('Number of Purchases per User')
ax.set_ylabel('Number of Users')
ax.set_title(
    f'Distribution of Purchase Counts per User\n'
    f'(values capped at {MAX_DISPLAY}; '
    f'{n_single_purchase/len(pu_nonzero)*100:.1f}% of purchase users have exactly 1 purchase)'
)
ax.set_xticks(x_vals)
ax.set_xticklabels(
    [str(x) if x < MAX_DISPLAY else f'>={MAX_DISPLAY}' for x in x_vals],
    rotation=45, ha='right', fontsize=9
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', linestyle='--', linewidth=0.4, alpha=0.4)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_user_purchase_sequence.pdf'))
plt.savefig(os.path.join(FIG_DIR, 'fig_user_purchase_sequence.png'))
plt.show()
print('Figure saved.')

## 5. Temporal Distribution of Interactions

Phan tich phan bo luong tuong tac theo tung thang de minh hoa giai doan thoi gian
cua tap du lieu va nen tang cho **Global Temporal Split**.

Du lieu CSV goc trong `data/raw/` duoc doc bang Pandas.


In [ ]:
# Doc cac file CSV goc va gop lai
csv_files = sorted([
    os.path.join(RAW_CSV_DIR, f)
    for f in os.listdir(RAW_CSV_DIR)
    if f.endswith('.csv')
])
print(f'Found {len(csv_files)} CSV files: {[os.path.basename(f) for f in csv_files]}')

dfs = []
for fp in csv_files:
    df_tmp = pd.read_csv(fp, usecols=['event_time', 'event_type'],
                         parse_dates=['event_time'])
    dfs.append(df_tmp)

df_all = pd.concat(dfs, ignore_index=True)
df_all['event_time'] = pd.to_datetime(df_all['event_time'], utc=True)
print(f'Tong so ban ghi: {len(df_all):,}')
print(df_all.dtypes)


In [ ]:
df_all['year_month'] = df_all['event_time'].dt.to_period('M').astype(str)
monthly_pd = (
    df_all.groupby(['year_month', 'event_type'])
    .size()
    .reset_index(name='count')
    .sort_values('year_month')
)
print(monthly_pd.to_string(index=False))


In [ ]:
# Ve bieu do stacked bar chart: phan bo tuong tac theo thang
pivot = monthly_pd.pivot(index='year_month', columns='event_type', values='count').fillna(0)
col_order = [c for c in ['view', 'cart', 'purchase'] if c in pivot.columns]
pivot = pivot[col_order]

colors_beh = {'view': '#2c6fad', 'cart': '#5ba4cf', 'purchase': '#f39c12'}

fig, ax = plt.subplots(figsize=(10, 4.5))
bottom = np.zeros(len(pivot))
for col in col_order:
    vals = pivot[col].values / 1e6
    ax.bar(
        pivot.index, vals, bottom=bottom,
        label=col.capitalize(),
        color=colors_beh.get(col, '#888888'),
        edgecolor='white', linewidth=0.4
    )
    bottom += vals

# Danh dau ranh gioi split
months = list(pivot.index)
train_end_month = '2020-01'
val_end_month   = '2020-02'

if train_end_month in months:
    split_x = months.index(train_end_month) + 0.5
    ax.axvline(split_x, color='black', linestyle='--', linewidth=1.0)
    ax.text(split_x + 0.05, bottom.max() * 1.02, 'Train | Val',
            fontsize=8.5, va='bottom', ha='left')

if val_end_month in months:
    split_x2 = months.index(val_end_month) + 0.5
    ax.axvline(split_x2, color='black', linestyle=':', linewidth=1.0)
    ax.text(split_x2 + 0.05, bottom.max() * 0.88, 'Val | Test',
            fontsize=8.5, va='bottom', ha='left')

ax.set_xlabel('Month')
ax.set_ylabel('Number of Interactions (millions)')
ax.set_title('Monthly Interaction Distribution by Behavior Type\n(Temporal Split Boundaries Marked)')
ax.legend(frameon=False, loc='upper left')
ax.set_xticks(range(len(months)))
ax.set_xticklabels(months, rotation=30, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_temporal_distribution.pdf'))
plt.savefig(os.path.join(FIG_DIR, 'fig_temporal_distribution.png'))
plt.show()
print('Figure saved.')


In [ ]:
# Kiem tra phan bo theo split (train/val/test) de xac nhan ty le phan chia
df_all['event_date'] = df_all['event_time'].dt.date.astype(str)
split_summary = df_all.copy()
split_summary['split'] = 'test'
split_summary.loc[split_summary['event_date'] <= '2020-01-31', 'split'] = 'train'
split_summary.loc[
    (split_summary['event_date'] > '2020-01-31') &
    (split_summary['event_date'] <= '2020-02-29'),
    'split'
] = 'val'

split_summary = (
    split_summary.groupby(['split', 'event_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['split', 'event_type'])
)
split_summary['pct'] = (
    split_summary.groupby('event_type')['count']
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)
print(split_summary.to_string(index=False))

# Cleanup
del df_all, split_summary


In [ ]:
# (Spark section removed — replaced by pandas above)


In [ ]:
# placeholder


## 6. Summary Statistics Table (for Paper)

Bang tom tat tong hop, san sang dan vao phan "Dataset" cua bai bao khoa hoc.

In [ ]:
# Tinh cac chi so tong hop cuoi cung
avg_view_per_user     = E_view     / N_U
avg_cart_per_user     = E_cart     / N_U
avg_purchase_per_user = E_purchase / N_U
avg_all_per_user      = E_behavior_total / N_U

interaction_density_purchase = E_purchase / max_pairs  # mat do (1 - sparsity)
interaction_density_all      = E_behavior_total / max_pairs

# Ratio giua view:cart:purchase
r_view = E_view / E_purchase
r_cart = E_cart / E_purchase

print('=' * 64)
print('   REES46 HETEROGENEOUS GRAPH -- DATASET STATISTICS SUMMARY')
print('=' * 64)
# Fix lỗi nháy đơn bằng cách dùng nháy đôi bên ngoài f-string
print(f"   {'Metric':<40} {'Value':>20}")
print('-' * 64)
print(f"   {'|U| Users':<40} {N_U:>20}")
print(f"   {'|I| Products':<40} {N_P:>20}")
print(f"   {'|C| Categories':<40} {N_C:>20}")
print(f"   {'|B| Brands':<40} {N_B:>20}")
print(f"   {'Total Nodes':<40} {N_total:>20}")
print('-' * 64)
print(f"   {'|E_view|':<40} {E_view:>20}")
print(f"   {'|E_cart|':<40} {E_cart:>20}")
print(f"   {'|E_purchase|':<40} {E_purchase:>20}")
print(f"   {'|E_belongsTo|':<40} {E_belongsTo:>20}")
print(f"   {'|E_producedBy|':<40} {E_producedBy:>20}")
print(f"   {'Total Behavioral Edges':<40} {E_behavior_total:>20}")
print('-' * 64)
print(f"   {'Sparsity (view)':<40} {sparsity_view:>20}")
print(f"   {'Sparsity (cart)':<40} {sparsity_cart:>20}")
print(f"   {'Sparsity (purchase)':<40} {sparsity_purchase:>20}")
print('-' * 64)
print(f"   {'Avg interactions / user':<40} {avg_all_per_user:>20}")
print(f"   {'Avg purchases / user':<40} {avg_purchase_per_user:>20}")
print(f"   {'view : purchase ratio':<40} {r_view:>20}")
print(f"   {'cart : purchase ratio':<40} {r_cart:>20}")
print('-' * 64)
print(f"   {'Temporal Val pairs':<40} {n_val:>20}")
print(f"   {'Temporal Test pairs':<40} {n_test:>20}")
print(f"   {'Unique eval users (val)':<40} {eval_users_val:>20}")
print(f"   {'Unique eval users (test)':<40} {eval_users_test:>20}")
print('=' * 64)